# RLHF v2: kuchliroq reward model + RLOO, SFT v3 LoRA (Qwen2.5-1.5B) ustida

RLHF v1 natijasi 40/68 (DPO v2: 44/68). Sabab PPO emas, **reward model** edi: qoida bali bilan korrelyatsiyasi +0.08,
uzunlik bilan −0.63 — u "qisqa javob yaxshi" deb o'rgangan, policy ham javoblarni 30% qisqartirgan.

v2'dagi o'zgarishlar:

| | v1 | v2 |
|---|---|---|
| Reward model data | 339 juftlik (har promptdan 1 ta) | ~1760 juftlik: 8 ta javobning **barcha** juftliklari (`rlhf/build_rm_data.py`) |
| Uzunlik muvozanati | yo'q | har bir vazifada chosen qisqa = chosen uzun (50/50) |
| Validation | 18 juftlik | ~180 juftlik, **promptlar** bo'yicha ajratilgan (78 prompt) |
| Reward model | Qwen2.5-0.5B base | Qwen2.5-1.5B + **SFT v3** (vazifalarni allaqachon biladi) |
| Uzunlik | yo'q | **length-controlled reward**: ball − `length_coef` × belgilar soni (qiyalik train'da, tekshiruv val'da) |
| RL algoritmi | PPO + value model | **RLOO**: har promptga 4 ta javob, baseline = qolgan 3 tasining o'rtachasi, value model yo'q |
| KL koeffitsienti | 0.05 | 0.1 (reward model hali ham mukammal emas) |

**Boshlashdan oldin:**

- **Settings -> Accelerator -> GPU T4 x2**, **Internet -> On**
- **SFT v3 adapter:** **Add Input** orqali `sft-v3-adapters` datasetini qo'shing (v1 va DPO'dagi bilan bir xil).
- **Save Version -> Save & Run All (Commit)**. Taxminiy vaqt: reward model ~30 daqiqa, RLOO ~3-4 soat, baholash ~10 daqiqa.
- Notebook **endi to'xtamaydi**: uzunlik tekshiruvi o'tmasa ham ogohlantirish chiqarib, oxirigacha ishlaydi.

## 1. GPU borligini tekshirish

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Loyiha va base modelni yuklab olish

v2'da reward model ham 1.5B, shuning uchun 0.5B kerak emas.

In [ ]:
%cd /kaggle/working
!rm -rf Post_Training
!GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/davronbek-malikov/Post_Training.git
%cd Post_Training
!pip -q install -U transformers peft bitsandbytes accelerate
!pip -q uninstall -y torchao  # Kaggle'dagi eski torchao peft bilan adapter yuklashni buzadi

from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen2.5-1.5B", local_dir="Qwen2.5-1.5B")
!ls -lh Qwen2.5-1.5B/*.safetensors

## 3. SFT v3 LoRA adapter

Policy, reference va endi reward model ham shundan boshlanadi.

In [ ]:
import glob, os, shutil, subprocess, zipfile
target = "chat_ui/adapters/sft-lora-v3"
found = glob.glob("/kaggle/input/**/sft-lora-v3/adapter_config.json", recursive=True)
if not found:   # zip ochilmagan holda yuklangan bo'lsa
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        with zipfile.ZipFile(z) as f:
            names = [n for n in f.namelist() if "sft-lora-v3/" in n]
            if names:
                f.extractall("/kaggle/working/unzipped", members=names)
    found = glob.glob("/kaggle/working/unzipped/**/sft-lora-v3/adapter_config.json", recursive=True)
if found:
    shutil.copytree(os.path.dirname(found[0]), target, dirs_exist_ok=True)
    print("SFT v3 adapter topildi:", os.path.dirname(found[0]))
else:
    print("Adapter topilmadi: SFT v3 LoRA qaytadan o'qitiladi...")
    subprocess.run(["python", "sft_v2/train.py", "--method", "lora", "--data", "sft_v3/data",
                    "--name", "sft-lora-v3", "--epochs", "3", "--rank", "64"], check=True)
print(os.listdir(target))

## 4. Reward model uchun data

Har bir promptdagi 8 ta javob va namunaviy javob `dpo/label.py` qoidalari bilan baholanadi. Balli kamida 0.5 farq
qiladigan har bir juftlik olinadi. Kuzating: rad etishsiz vazifalarda **chosen qisqaroq ≈ 50%** bo'lishi kerak.

In [ ]:
!python rlhf/build_rm_data.py

## 5. Reward model (Qwen2.5-1.5B + SFT v3 + LoRA)

Tekshiring:
- `val acc` (v1: 0.72, 18 juftlikda) — endi ~180 juftlikda, natija ancha ishonchli;
- `val acc [vazifa]` — reward model qaysi vazifada kuchsiz;
- `Uzunlik tekshiruvi, tuzatishsiz` — oldingi run'da pair_corr -0.35 (qisqa javobga kattaroq farq beradi);
- `Uzunlik tuzatishi` — har bir belgi uchun ball qancha o'zgaradi (train juftliklaridan);
- **tuzatilgan** uzunlik tekshiruvi `ok: True` bo'lishi kerak. `False` bo'lsa ham notebook davom etadi, lekin natijaga
  ehtiyot bo'lib qaraladi.

1 epoch: oldingi run'da 2-epoch faqat overfit qildi (val loss 0.58 -> 0.82), eng yaxshi model 1-epochdan edi.

In [ ]:
!python rlhf/train_reward.py --base Qwen2.5-1.5B --sft-adapter sft-lora-v3 --data rlhf/data_v2 --out 2_sft-1.5b --epochs 1

In [ ]:
import json
meta = json.load(open("rlhf/models/2_sft-1.5b/meta.json"))
print(json.dumps(meta, indent=1))
# Skript yarmida xato bilan to'xtagan bo'lsa, length_gate yozilmagan bo'ladi: aniq xato beramiz.
assert "length_gate" in meta, "train_reward.py oxirigacha ishlamadi: yuqoridagi katak chiqishidagi xatoni qarang"
if meta["length_gate"]["ok"]:
    print("Uzunlik tekshiruvi o'tdi: RLOO boshlanadi")
else:
    print("OGOHLANTIRISH: uzunlik tekshiruvi tuzatishdan keyin ham o'tmadi. RLOO baribir ishlaydi (--force),"
          " lekin natijada javob uzunligini alohida tekshiring.")

## 6. RLOO

Har qadamda 8 ta prompt × 4 ta javob = 32 ta rollout. Kuzating:
- 1-qadamda `KL` ~0;
- `RM ball` va **`qoida bali` birga o'sishi kerak**. v1'da RM ball o'sdi, qoida bali tushdi (0.62 -> 0.57) — reward hacking;
- `uzunlik` keskin tushmasligi kerak (v1: 31 -> 28 token, test javoblari 30% qisqardi);
- `vf` doim 0: value model yo'q.

In [ ]:
!python rlhf/train_ppo.py --algo rloo --group-size 4 --batch-size 8 --mini-batch 4 \
    --reward 2_sft-1.5b --reward-fp16 --kl-coef 0.1 \
    --name sft-lora-rlhf-v2 --rollouts rlhf_v2_rollouts.jsonl --force
import os
assert os.path.exists("chat_ui/adapters/sft-lora-rlhf-v2/adapter_config.json"), 'RLOO adapterni saqlamadi: yuqoridagi xatoni qarang'

In [ ]:
!rm -f /kaggle/working/rlhf_v2_results.zip
!zip -q -r /kaggle/working/rlhf_v2_results.zip chat_ui/adapters/sft-lora-rlhf-v2 rlhf/logs/rlhf_v2_rollouts.jsonl rlhf/models/2_sft-1.5b/meta.json rlhf/models/2_sft-1.5b/training_log.json
!ls -lh /kaggle/working/rlhf_v2_results.zip

## 7. Baholash: 68 ta test savoli

SFT v3: 41/68. DPO v2: 44/68. RLHF v1: 40/68.

In [ ]:
!python eval/run_eval.py --base Qwen2.5-1.5B --model sft-lora-rlhf-v2 | tail -12
!zip -q /kaggle/working/rlhf_v2_results.zip eval/results/sft-lora-rlhf-v2.json
assert os.path.exists("eval/results/sft-lora-rlhf-v2.json"), 'Baholash ishlamadi: yuqoridagi xatoni qarang'

## 8. Yuklab olish

Notebook tugagach: **Output** -> `rlhf_v2_results.zip` -> **Download**. Uni loyihaning `rlhf/` papkasiga qo'ying va
loyiha ildizida oching (ichidagi yo'llar `chat_ui/...`, `rlhf/...`, `eval/...` dan boshlanadi). Keyin
`python eval/summarize.py` — jadvalga **v3 + RLHF v2** ustuni qo'shiladi, chat UI'da model yonadi.